# Character Swap Augmentation for Spam Detection

This notebook demonstrates how to apply character-level noise (specifically character swaps) to spam messages in a dataset. This is useful for creating robust models that can handle obfuscated spam content.

## Install Dependencies

We begin by installing necessary libraries:  
- `nlpaug` for data augmentation  
- `swifter` for faster row-wise DataFrame operations  
- `tqdm` for progress bars in notebooks

In [1]:
!pip install nlpaug swifter tqdm

     ---------------------------------------- 1.2/1.2 MB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16519 sha256=e2952d2a579ebc4bc9f13799f60243c7819c69bc2da3fe7254811653f827e791
  Stored in directory: c:\users\butit\appdata\local\pip\cache\wheels\43\a7\a3\1194ca51c35c2a0c0041c97e4a9c1f0ed82a20cb3b1b08d610
Successfully built swifter


## Import Libraries

We import required libraries including Pandas, Swifter, and `nlpaug`'s character augmentation module.

In [1]:
import pandas as pd
import swifter
from tqdm.notebook import tqdm
import nlpaug.augmenter.char as nac

## Load Dataset

We load a training dataset of SMS messages. The dataset is assumed to be in `../dataset/sms/train.csv`. This file should contain two columns:  
- `email`: the message text  
- `target`: a label indicating whether the message is spam or ham

In [2]:
# Load dataset
df = pd.read_csv("../dataset/sms/train.csv")
df.head()

,email,target
0,What to think no one saying clearly. Ok leave ...,ham
1,"FREE RING TONE just text \POLYS\"" to 87131. Th...",spam
2,"Trust me. Even if isn't there, its there.",ham
3,Hi dear we saw dear. We both are happy. Where ...,ham
4,"URGENT, IMPORTANT INFORMATION FOR O2 USER. TOD...",spam


## Inspect Dataset

Let's examine the structure of the dataset to ensure it contains the expected data types and no missing values.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3565 entries, 0 to 3564
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   email   3565 non-null   object
 1   target  3565 non-null   object
dtypes: object(2)
memory usage: 55.8+ KB


## Create Character Swap Augmenter

We use `nlpaug`'s `RandomCharAug` with the `swap` action to randomly swap characters in a message, simulating typos or obfuscation.

In [4]:
# Create a character swap augmenter with a moderate swap probability
char_aug = nac.RandomCharAug(action="swap", aug_char_p=0.20)

## Define Augmentation Function

This function takes a row and applies character swaps to the `email` text **only if it is labeled as spam**.

In [5]:
def augment_if_spam(row):
    """
    Applies a character-level swap attack using nlpaug to spam emails only.

    Parameters:
        row (pd.Series): A row from the DataFrame with 'email' and 'target'.

    Returns:
        pd.Series: (augmented_text, was_augmented)
    """
    email = row.get('email', '')

    # Check if the row is labeled as spam AND the 'email' field is a non-empty string
    if row.get('target') == 'spam' and isinstance(email, str) and email.strip():
        try:
            # Apply character-level augmentation using nlpaug
            augmented = char_aug.augment(email)
            # Return the augmented text and a flag indicating it was modified
            return pd.Series([augmented, True])
        except Exception as e:
            # If augmentation fails (e.g., due to unexpected input), print the error
            print(f"Augmentation failed: {e}")
            # Return the original email and a flag indicating it was not modified
            return pd.Series([email, False])
    else:
        # For non-spam entries, return the original email without changes
        return pd.Series([email, False])

## Apply Augmentation to Dataset

We apply the `augment_if_spam` function across all rows using `swifter` to parallelize the process, and use `tqdm` to visualize progress.

In [6]:
tqdm.pandas()  # Enables progress bar
df[['email_charswapped', 'was_augmented']] = df.swifter.apply(augment_if_spam, axis=1)

Pandas Apply:   0%|          | 0/3565 [00:00<?, ?it/s]

## Preview Augmented Data

We display the first few rows of the dataset, now including a new column `email_charswapped` and a flag `was_augmented`.

In [7]:
df.head(10)

,email,target,email_charswapped,was_augmented
0,What to think no one saying clearly. Ok leave ...,ham,What to think no one saying clearly. Ok leave ...,False
1,"FREE RING TONE just text \POLYS\"" to 87131. Th...",spam,"[FREE RIGN TONE juts etxt \ POLYS \ "" to 87113...",True
2,"Trust me. Even if isn't there, its there.",ham,"Trust me. Even if isn't there, its there.",False
3,Hi dear we saw dear. We both are happy. Where ...,ham,Hi dear we saw dear. We both are happy. Where ...,False
4,"URGENT, IMPORTANT INFORMATION FOR O2 USER. TOD...",spam,"[URNGET, IMPORTANT INFORMATION FOR O2 USER. OT...",True
5,Yeah there's barely enough room for the two of...,ham,Yeah there's barely enough room for the two of...,False
6,Jus telling u dat i'll b leaving 4 shanghai on...,ham,Jus telling u dat i'll b leaving 4 shanghai on...,False
7,Am slow in using biola's fne,ham,Am slow in using biola's fne,False
8,Your gonna be the death if me. I'm gonna leave...,ham,Your gonna be the death if me. I'm gonna leave...,False
9,"Good morning, my boytoy! How's those yummy lip...",ham,"Good morning, my boytoy! How's those yummy lip...",False


## Save Augmented Dataset

Finally, we save the modified DataFrame to a new CSV file for downstream use in model training or analysis.

In [9]:
df.to_csv("../dataset/sms/charswap/train_with_charswap.csv", index=False)